In [2]:
# 1. Install dependencies
!pip install -q datasets huggingface_hub opencv-python-headless torch torchvision torchaudio \
    pytorchvideo av decord tqdm matplotlib ultralytics scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 12.7 MB/s eta 0:00:00


In [1]:
from datasets import load_dataset

# Downloads to ~/.cache/huggingface/datasets the first time you run it
ds = load_dataset("DanJoshua/RWF-2000")
print(ds)
print(ds["train"][0].keys())

README.md:   0%|          | 0.00/4.12k [00:00<?, ?B/s]

RWF-2000.tar.gz: reconstructing file:   0%|          |  0.00B / 12.3GB            

RWF-2000.tar.gz: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['avi', '__key__', '__url__'],
        num_rows: 2000
    })
})
dict_keys(['avi', '__key__', '__url__'])


In [3]:
# Quick peek at a few samples (works once ds is loaded)
import matplotlib.pyplot as plt

sample = ds["train"][0]
print(sample)
# If the dataset stores a video path/bytes, adjust the key name printed above accordingly


{'avi': b'RIFF2\x9f%\x00AVI LIST\xec\x11\x00\x00hdrlavih8\x00\x00\x005\x82\x00\x00\x00\x1aO\x00\x00\x00\x00\x00\x10\t\x00\x00\x96\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x10\x00\x80\x02\x00\x00h\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00LIST\x94\x10\x00\x00strlstrh8\x00\x00\x00vidsMJPG\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x1e\x00\x00\x00\x00\x00\x00\x00\x96\x00\x00\x00hA\x00\x00\xff\xff\xff\xff\x00\x00\x00\x00\x00\x00\x00\x00\x80\x02h\x01strf(\x00\x00\x00(\x00\x00\x00\x80\x02\x00\x00h\x01\x00\x00\x01\x00\x18\x00MJPG\x00\x8c\n\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00JUNK\x18\x10\x00\x00\x04\x00\x00\x00\x00\x00\x00\x0000dc\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\

In [4]:
from collections import Counter

if "label" in ds["train"].features:
    labels = ds["train"]["label"]
    print(Counter(labels))


In [5]:
import os, glob, shutil, math
import cv2
import matplotlib.pyplot as plt

RAW_DIR = "path/to/your_10k_images"      # <-- set this
FILTERED_DIR = "path/to/filtered_people"  # after step A (person-count filter)
KEEP_DIR = "path/to/final_fight_dataset"  # your curated output
os.makedirs(KEEP_DIR, exist_ok=True)

def filter_by_person_count(src_dir, dst_dir, min_people=2):
    """Step A: keep only images where a YOLO person-detector finds >= min_people."""
    from ultralytics import YOLO
    model = YOLO("yolov8n.pt")  # tiny + fast, good enough for a pre-filter
    os.makedirs(dst_dir, exist_ok=True)
    kept = 0
    for img_path in glob.glob(os.path.join(src_dir, "*")):
        results = model(img_path, classes=[0], verbose=False)  # class 0 = person in COCO
        n_people = len(results[0].boxes)
        if n_people >= min_people:
            shutil.copy(img_path, dst_dir)
            kept += 1
    print(f"Kept {kept} images with >= {min_people} people")

def review_grid(image_dir, batch_size=12, thumb_size=200):
    """Step B: show images in batches, type the indices (comma separated) of the ones to KEEP."""
    paths = sorted(glob.glob(os.path.join(image_dir, "*")))
    for start in range(0, len(paths), batch_size):
        batch = paths[start:start + batch_size]
        cols = 4
        rows = math.ceil(len(batch) / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
        axes = axes.flatten()
        for i, p in enumerate(batch):
            img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
            axes[i].imshow(img)
            axes[i].set_title(str(i))
            axes[i].axis("off")
        for j in range(len(batch), len(axes)):
            axes[j].axis("off")
        plt.tight_layout()
        plt.show()

        keep_idx = input(f"Batch {start}-{start+len(batch)}: indices to KEEP (comma-separated, blank = none): ")
        for idx in keep_idx.split(","):
            idx = idx.strip()
            if idx.isdigit():
                shutil.copy(batch[int(idx)], KEEP_DIR)

# Usage:
# filter_by_person_count(RAW_DIR, FILTERED_DIR)
# review_grid(FILTERED_DIR)


In [6]:
import torch

# Example: load a Kinetics-400-pretrained X3D model via PyTorchVideo, ready to fine-tune
model = torch.hub.load("facebookresearch/pytorchvideo", "x3d_s", pretrained=True)

# Replace the final classification head for binary fight / no-fight
in_features = model.blocks[-1].proj.in_features
model.blocks[-1].proj = torch.nn.Linear(in_features, 2)  # 0=no_fight, 1=fight
print(model.blocks[-1])


Downloading: "https://github.com/facebookresearch/pytorchvideo/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/pytorchvideo/model_zoo/kinetics/X3D_S.pyth" to /root/.cache/torch/hub/checkpoints/X3D_S.pyth


100%|██████████| 29.4M/29.4M [00:00<00:00, 210MB/s]


ResNetBasicHead(
  (pool): ProjectedPool(
    (pre_conv): Conv3d(192, 432, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
    (pre_norm): BatchNorm3d(432, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (pre_act): ReLU()
    (pool): AvgPool3d(kernel_size=(np.int64(13), 5, 5), stride=1, padding=0)
    (post_conv): Conv3d(432, 2048, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
    (post_act): ReLU()
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (proj): Linear(in_features=2048, out_features=2, bias=True)
  (output_pool): AdaptiveAvgPool3d(output_size=1)
)


In [7]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

class FightClipDataset(Dataset):
    """TODO: point this at your combined, curated dataset (RWF-2000 + extras)."""
    def __init__(self, clip_paths, labels, num_frames=16, transform=None):
        self.clip_paths = clip_paths
        self.labels = labels
        self.num_frames = num_frames
        self.transform = transform

    def __len__(self):
        return len(self.clip_paths)

    def __getitem__(self, idx):
        # TODO: use decord/opencv to read `num_frames` evenly spaced frames from clip_paths[idx],
        # stack into a [C, T, H, W] tensor, apply self.transform, return (clip, label)
        raise NotImplementedError

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for clips, labels in loader:
        clips, labels = clips.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(clips)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * clips.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += clips.size(0)
    return total_loss / total, correct / total

# TODO: build train_loader / val_loader from FightClipDataset, then:
# for epoch in range(20):
#     train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
#     scheduler.step()
#     print(f"Epoch {epoch}: loss={train_loss:.4f} acc={train_acc:.4f}")


In [8]:
from sklearn.metrics import classification_report, confusion_matrix

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for clips, labels in loader:
        clips = clips.to(device)
        preds = model(clips).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
    print(classification_report(all_labels, all_preds, target_names=["no_fight", "fight"]))
    print(confusion_matrix(all_labels, all_labels))

# evaluate(model, val_loader, device)

torch.save(model.state_dict(), "fight_detector_x3d_s.pt")


In [9]:
import cv2
import numpy as np
import torch
from collections import deque
from ultralytics import YOLO

person_detector = YOLO("yolov8n.pt")  # or a larger yolov8 model for better accuracy
fight_model = model  # the fine-tuned model from above, in eval mode
fight_model.eval()

NUM_FRAMES = 16
STRIDE = 8
CONF_THRESHOLD = 0.75

def send_alert(frame, confidence):
    """TODO: wire this up to whatever alerting channel you need — webhook, email, SMS, siren, etc."""
    print(f"[ALERT] Fight detected, confidence={confidence:.2f}")
    # e.g. requests.post("https://your-alert-endpoint/notify", json={"confidence": confidence})

def preprocess_clip(frames):
    """frames: list of BGR np arrays -> tensor [1, C, T, H, W]"""
    clip = np.stack([cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2RGB), (224, 224)) for f in frames])
    clip = torch.from_numpy(clip).float() / 255.0
    clip = clip.permute(3, 0, 1, 2).unsqueeze(0)  # [1, C, T, H, W]
    return clip

def run_stream(source=0):
    """source=0 for webcam, or a video file path / RTSP URL for a CCTV feed."""
    cap = cv2.VideoCapture(source)
    buffer = deque(maxlen=NUM_FRAMES)
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        buffer.append(frame)
        frame_count += 1

        # People detection every frame (for drawing boxes)
        results = person_detector(frame, classes=[0], verbose=False)
        boxes = results[0].boxes.xyxy.cpu().numpy() if len(results[0].boxes) else []

        is_fight, confidence = False, 0.0
        if len(buffer) == NUM_FRAMES and frame_count % STRIDE == 0:
            clip = preprocess_clip(list(buffer)).to(device)
            with torch.no_grad():
                probs = torch.softmax(fight_model(clip), dim=1)[0]
                confidence = probs[1].item()  # index 1 = "fight"
                is_fight = confidence >= CONF_THRESHOLD

        color = (0, 0, 255) if is_fight else (0, 255, 0)
        for box in boxes:
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        if is_fight:
            cv2.putText(frame, f"FIGHT DETECTED ({confidence:.2f})", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            send_alert(frame, confidence)

        cv2.imshow("Fight Detection", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

# run_stream(0)              # webcam
# run_stream("test.mp4")     # video file
# run_stream("rtsp://...")   # live CCTV/IP camera feed


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
